# AniMerPlus Demo

This notebook provides an interactive way to run `demo.py` and preview rendered results. Update the parameters below and execute the cells in order.

In [1]:
from pathlib import Path
import detectron2.config
import detectron2.engine
import torch
import argparse
import os
import cv2
import numpy as np
from tqdm import tqdm
import torch.utils
import torch.utils.data
from amr.models import load_amr
from amr.utils import recursive_to
from amr.datasets.vitdet_dataset import ViTDetDataset, DEFAULT_MEAN, DEFAULT_STD
from amr.utils.renderer import Renderer, cam_crop_to_full
import detectron2
from detectron2 import model_zoo
import warnings
warnings.filterwarnings("ignore")

LIGHT_BLUE = (0.65098039, 0.74117647, 0.85882353)

In [5]:
checkpoint = 'data/AniMerPlus/checkpoint.ckpt'
img_folder = 'horse-data'
out_folder = 'demo_out'
side_view = False
save_mesh = False
batch_size = 1
file_type = ['*.jpg', '*.png', '*JPEG', '*.jpeg']
animal_type = 'bird'  # or 'mammal'

print('checkpoint =', checkpoint)
print('img_folder =', img_folder)
print('out_folder =', out_folder)
print('side_view =', side_view)
print('save_mesh =', save_mesh)
print('batch_size =', batch_size)
print('file_type =', file_type)
print('animal_type =', animal_type)

checkpoint = data/AniMerPlus/checkpoint.ckpt
img_folder = horse-data
out_folder = demo_out
side_view = False
save_mesh = False
batch_size = 1
file_type = ['*.jpg', '*.png', '*JPEG', '*.jpeg']
animal_type = bird


In [12]:
import os
os.environ["PYOPENGL_PLATFORM"] = "egl"
model, model_cfg = load_amr(checkpoint)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = model.to(device)
model.eval()

# Setup the renderer
if animal_type == "bird":
    renderer = Renderer(model_cfg, faces=model.aves.face)
    coco_category = [14]
elif animal_type == "mammal":
    renderer = Renderer(model_cfg, faces=model.smal.faces)
    coco_category = [15, 16, 17, 18, 19, 21, 22]
else:
    raise ValueError("Unsupported animal type. Choose 'mammal' or 'bird'.")

# Make output directory if it does not exist
os.makedirs(out_folder, exist_ok=True)

print("Loading model and detector...")
# Load detector
cfg = detectron2.config.get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5 
cfg.MODEL.WEIGHTS = "https://dl.fbaipublicfiles.com/detectron2/COCO-Detection/faster_rcnn_X_101_32x8d_FPN_3x/139173657/model_final_68b088.pkl"
detector = detectron2.engine.DefaultPredictor(cfg)
img_paths = sorted([img for end in file_type for img in Path(img_folder).glob(end)])
print(f"Found {len(img_paths)} images in {img_folder} with extensions {file_type}")
for img_path in img_paths:
    print("Processing image:", img_path)
    img_cv2 = cv2.imread(str(img_path))
    img_cv2 = cv2.cvtColor(img_cv2, cv2.COLOR_BGR2RGB)

    # Detect humans in image
    det_out = detector(img_cv2)

    det_instances = det_out['instances']
    # valid_idx = [i for i, (c, s) in enumerate(zip(det_instances.pred_classes, det_instances.scores)) if ((c in coco_category) & (s > 0.8))]
    # if len(valid_idx) == 0:
    #     print(f"No valid detections found in {img_path}. Skipping...")
    #     continue
    valid_idx = [0]
    category = det_instances.pred_classes[valid_idx].cpu().numpy()
    category = np.where(category==14, 17, 0)  # 6 for birds
    boxes = det_instances.pred_boxes.tensor[valid_idx].cpu().numpy()

    # Run AniMer on detected animals
    dataset = ViTDetDataset(model_cfg, img_cv2, boxes, category)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
    for batch in tqdm(dataloader):
        batch = recursive_to(batch, device)
        with torch.no_grad():
            out = model(batch)
            if "aves_output" in out or "smal_output" in out:
                out = out["aves_output"] if "aves_output" in out else out["smal_output"]

        pred_cam = out['pred_cam']
        box_center = batch["box_center"].float()
        box_size = batch["box_size"].float()
        img_size = batch["img_size"].float()
        scaled_focal_length = renderer.focal_length / model_cfg.MODEL.IMAGE_SIZE * img_size.max()
        pred_cam_t_full = cam_crop_to_full(pred_cam, box_center, box_size, img_size,
                                            scaled_focal_length).detach().cpu().numpy()

        # Render the result
        batch_size = batch['img'].shape[0]
        for n in range(batch_size):
            # Get filename from path img_path
            img_fn, _ = os.path.splitext(os.path.basename(img_path))
            animal_id = int(batch['animalid'][n])
            white_img = (torch.ones_like(batch['img'][n]).cpu() - DEFAULT_MEAN[:, None, None] / 255) / (
                        DEFAULT_STD[:, None, None] / 255)
            input_patch = (batch['img'][n].cpu() * (DEFAULT_STD[:, None, None]) + (
                        DEFAULT_MEAN[:, None, None])) / 255.
            input_patch = input_patch.permute(1, 2, 0).numpy()

            # regression_img = renderer(out['pred_vertices'][n].detach().cpu().numpy(),
            #                         out['pred_cam_t'][n].detach().cpu().numpy(),
            #                         batch['img'][n],
            #                         mesh_base_color=LIGHT_BLUE,
            #                         scene_bg_color=(1, 1, 1),
            #                             )

            # final_img = np.concatenate([input_patch, regression_img], axis=1)

            # cv2.imwrite(os.path.join(out_folder, f'{img_fn}_{animal_id}.png'), 
            #             cv2.cvtColor((255 * final_img[:, :, ::-1]).astype(np.uint8), cv2.COLOR_RGB2BGR))

            # Add all verts and cams to list
            verts = out['pred_vertices'][n].detach().cpu().numpy()
            cam_t = pred_cam_t_full[n]
            print("Processed animal ID:", animal_id)

            # Save all meshes to disk
            if True:
                print(f"Saving mesh for {img_fn}_{animal_id}.obj")
                camera_translation = cam_t.copy()
                tmesh = renderer.vertices_to_trimesh(verts, camera_translation, LIGHT_BLUE)
                tmesh.export(os.path.join(out_folder, f'{img_fn}_{animal_id}.obj'))


Loading model and detector...
Found 1 images in horse-data with extensions ['*.jpg', '*.png', '*JPEG', '*.jpeg']
Processing image: horse-data/Horse_Image.jpg


  0%|          | 0/1 [00:00<?, ?it/s]

downsampling_factor=np.float32(1.5298876)


  0%|          | 0/1 [00:00<?, ?it/s]

Processed animal ID: 0
Saving mesh for Horse_Image_0.obj


IndexError: index 4097 is out of bounds for axis 0 with size 3889

In [7]:
det_out

{'instances': Instances(num_instances=1, image_height=360, image_width=539, fields=[pred_boxes: Boxes(tensor([[124.8661,  56.9204, 418.6045, 320.6506]], device='cuda:0')), scores: tensor([0.9997], device='cuda:0'), pred_classes: tensor([17], device='cuda:0')])}

In [1]:
import os
from pathlib import Path
import sys

# Ensure the notebook runs from the AniMerPlus package root
root_dir = Path('/home/om/mpi/AniMerPlus')
os.chdir(root_dir)
sys.path.insert(0, str(root_dir))

print('Working directory:', os.getcwd())
print('Python path includes:', sys.path[0])

Working directory: /home/om/mpi/AniMerPlus
Python path includes: /home/om/mpi/AniMerPlus


## Parameters
Edit these values before running the demo.

In [2]:
checkpoint = 'data/AniMerPlus/checkpoint.ckpt'
img_folder = 'example_data'
out_folder = 'demo_out'
side_view = False
save_mesh = False
batch_size = 1
file_type = ['*.jpg', '*.png', '*JPEG', '*.jpeg']
animal_type = 'bird'  # or 'mammal'

print('checkpoint =', checkpoint)
print('img_folder =', img_folder)
print('out_folder =', out_folder)
print('side_view =', side_view)
print('save_mesh =', save_mesh)
print('batch_size =', batch_size)
print('file_type =', file_type)
print('animal_type =', animal_type)

checkpoint = data/AniMerPlus/checkpoint.ckpt
img_folder = example_data
out_folder = demo_out
side_view = False
save_mesh = False
batch_size = 1
file_type = ['*.jpg', '*.png', '*JPEG', '*.jpeg']
animal_type = bird


## Run Demo Script
This cell runs the existing `demo.py` script with the parameters above.

In [3]:
from IPython.display import display, Markdown

args = [
    'python', 'demo.py',
    '--checkpoint', checkpoint,
    '--img_folder', img_folder,
    '--out_folder', out_folder,
    '--batch_size', str(batch_size),
    '--animal_type', animal_type,
]

if side_view:
    args.append('--side_view')
if save_mesh:
    args.append('--save_mesh')
for pattern in file_type:
    args.extend(['--file_type', pattern])

print('Running:', ' '.join(args))

import subprocess
result = subprocess.run(args, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'Demo script failed with exit code {result.returncode}')

display(Markdown('**Demo completed successfully.**'))

Running: python demo.py --checkpoint data/AniMerPlus/checkpoint.ckpt --img_folder example_data --out_folder demo_out --batch_size 1 --animal_type bird --file_type *.jpg --file_type *.png --file_type *JPEG --file_type *.jpeg

libEGL warning: pci id for fd 17: 10de:1b02, driver (null)

pci id for fd 18: 10de:1b02, driver (null)
pci id for fd 19: 10de:1b02, driver (null)
libEGL warning: egl: failed to create dri2 screen
libEGL warning: pci id for fd 17: 10de:1b02, driver (null)

pci id for fd 18: 10de:1b02, driver (null)
pci id for fd 19: 10de:1b02, driver (null)
libEGL warning: egl: failed to create dri2 screen
libEGL warning: pci id for fd 17: 10de:1b02, driver (null)

Traceback (most recent call last):
  File "/home/om/mpi/AniMerPlus/demo.py", line 154, in <module>
    main()
  File "/home/om/mpi/AniMerPlus/demo.py", line 44, in main
    model, model_cfg = load_amr(args.checkpoint)
  File "/home/om/mpi/AniMerPlus/amr/models/__init__.py", line 26, in load_amr
    model = AniMerPlusPlus.

RuntimeError: Demo script failed with exit code 1

## Preview Output Images
The following cell displays rendered images from the output folder.

In [ ]:
from IPython.display import Image, display

output_path = Path(out_folder)
images = sorted(output_path.glob('*.png'))

if not images:
    print('No output images found in', output_path)
else:
    for image_path in images:
        print(image_path.name)
        display(Image(filename=str(image_path)))